# Klayout 学习测试
DBU：database Unit 数据库单位  um


In [31]:
import klayout.db as kdb

layout = kdb.Layout()
layout.read("../TestReticle/test1.gds")
print("DBU=",layout.dbu)
tops = layout.top_cells()

for cell in tops:
    print("TOP CELL:", cell.name)

DBU= 0.001
TOP CELL: test
TOP CELL: cell1


## 读取Layer
layer_index 是klayout内部使用的index，和layer、datatype不一样；


In [33]:
import klayout.db as kdb

layout = kdb.Layout()
layout.read("../TestReticle/test1.gds")

top = layout.top_cells()[1]
layer_index = layout.find_layer(1, 0)

print("Layer index:", layer_index)

for layer_index in layout.layer_indexes():
    info = layout.get_info(layer_index)
    print(
        "index:",
        layer_index,
        "layer:",
        info.layer,
        "datatype:",
        info.datatype,
        "name:",
        info.name
    )

Layer index: 1
index: 0 layer: 2 datatype: 0 name: 
index: 1 layer: 1 datatype: 0 name: 
index: 2 layer: 3 datatype: 0 name: 


在 KLayout 中，Shape 对象中的几何坐标通常存储在其所属 cell 的局部坐标系（local coordinate system） 下。当通过 begin_shapes_rec() 递归遍历层级结构时，获取到的 Shape 可能位于子 cell 内部，而该子 cell 又可能通过平移、旋转、镜像等方式实例化到上层 cell 中。因此，直接读取 shape.polygon 得到的坐标并不一定对应整个版图的全局位置。

为了获得顶层版图（TOP cell）坐标系下的真实几何位置，需要利用递归迭代器返回的变换矩阵 it.trans() 对几何对象进行坐标转换。该变换记录了当前 Shape 从所属 cell 到 TOP cell 的空间变换关系，包括平移（translation）、旋转（rotation）以及镜像（reflection）等操作。通过调用 polygon.transformed(it.trans())，可以将原本位于局部 cell 坐标系中的 polygon 转换到统一的 TOP 坐标系中，使所有图形具有一致的参考坐标。

完成坐标转换后，得到的 polygon 顶点坐标仍然是 GDS 内部数据库单位（database unit, DBU），需要进一步乘以版图精度参数 layout.dbu 转换为实际物理尺寸。

box (-1000,-1000;1400,-800) r0 *1 3700,3200
box (-1000,-1000;1400,-800) 是 it.shape,表示的是这个形状的局部坐标；后面的是 it.trans(),r0指旋转0°，*1指放大倍数，3700，3200指全局的初始点坐标

In [37]:
import klayout.db as kdb

layout = kdb.Layout()
layout.read("../TestReticle/test1.gds")

dbu = layout.dbu
top = layout.top_cells()[0]
layer_index = layout.find_layer(1, 0)
iterator = top.begin_shapes_rec(layer_index)

polygon_id = 0
for it in iterator:
    shape = it.shape()
    print(shape, it.trans())

    if shape.is_polygon():
        poly = shape.polygon
    elif shape.is_box():
        poly = kdb.Polygon(shape.box)
    else:
        continue

    # 转换到 TOP cell 坐标
    poly = poly.transformed(it.trans())
    print("Polygon", polygon_id)

    for p in poly.each_point_hull():
        x_um = p.x * dbu
        y_um = p.y * dbu
        print(x_um, y_um)

    edge_id = 0
    for edge in poly.each_edge():
        print(f"Edge {edge_id}:({edge.p1.x * dbu},{edge.p1.y * dbu})--({edge.p2.x * dbu},{edge.p2.y * dbu})")
        edge_id += 1

    polygon_id += 1

simple_polygon (-99,-500;-283,-424;-424,-283;-500,-99;-500,99;-424,283;-283,424;-99,500;99,500;283,424;424,283;500,99;500,-99;424,-283;283,-424;99,-500) r0 *1 -900,500
Polygon 0
-0.999 0.0
-1.183 0.076
-1.324 0.217
-1.4000000000000001 0.401
-1.4000000000000001 0.599
-1.324 0.783
-1.183 0.924
-0.999 1.0
-0.801 1.0
-0.617 0.924
-0.47600000000000003 0.783
-0.4 0.599
-0.4 0.401
-0.47600000000000003 0.217
-0.617 0.076
-0.801 0.0
Edge 0:(-0.999,0.0)--(-1.183,0.076)
Edge 1:(-1.183,0.076)--(-1.324,0.217)
Edge 2:(-1.324,0.217)--(-1.4000000000000001,0.401)
Edge 3:(-1.4000000000000001,0.401)--(-1.4000000000000001,0.599)
Edge 4:(-1.4000000000000001,0.599)--(-1.324,0.783)
Edge 5:(-1.324,0.783)--(-1.183,0.924)
Edge 6:(-1.183,0.924)--(-0.999,1.0)
Edge 7:(-0.999,1.0)--(-0.801,1.0)
Edge 8:(-0.801,1.0)--(-0.617,0.924)
Edge 9:(-0.617,0.924)--(-0.47600000000000003,0.783)
Edge 10:(-0.47600000000000003,0.783)--(-0.4,0.599)
Edge 11:(-0.4,0.599)--(-0.4,0.401)
Edge 12:(-0.4,0.401)--(-0.47600000000000003,0.217)

In [65]:
import klayout.db as kdb

layout = kdb.Layout()
layout.read("../TestReticle/test1.gds")

dbu = layout.dbu

# 获取top cell
tops = layout.top_cells()
for cell in tops:
    print("TOP CELL:", cell.name)


# 获取layer
for layer_index in layout.layer_indexes():
    info = layout.get_info(layer_index)
    print(
        "layer index:",
        layer_index,
        "layer:",
        info.layer,
        "datatype:",
        info.datatype,
        "name:",
        info.name,
        "info:",
        info
    )

top = layout.top_cells()[1]
top_bbox = top.bbox()
print("TOP_BBOX:", top_bbox)
global_bbox = top.bbox_per_layer(2)
print("LAYER_TOP_BBOX:", global_bbox)

layer_index = layout.find_layer(3, 0)
iterator = top.begin_shapes_rec(layer_index)

# 迭代某个区域
new_bbox = kdb.Box((top_bbox.left + (top_bbox.right - top_bbox.left) / 4, top_bbox.bottom + (top_bbox.top - top_bbox.bottom) / 4),
                      (top_bbox.right - (top_bbox.right - top_bbox.left) / 4, top_bbox.top - (top_bbox.top - top_bbox.bottom) / 4))
# print(new_bbox)
iterator = top.begin_shapes_rec_overlapping(layer_index, new_bbox)


polygon_id = 0
for it in iterator:
    shape = it.shape()
    print(shape, it.trans())

    if shape.is_polygon():
        poly = shape.polygon
    elif shape.is_box():
        poly = kdb.Polygon(shape.box)
    else:
        continue

    # 转换到 TOP cell 坐标
    poly = poly.transformed(it.trans())
    print("Polygon", polygon_id)

    for p in poly.each_point_hull():
        x_um = p.x * dbu
        y_um = p.y * dbu
        print(x_um, y_um)

    edge_id = 0
    for edge in poly.each_edge():
        print(f"Edge {edge_id}:({edge.p1.x * dbu},{edge.p1.y * dbu})--({edge.p2.x * dbu},{edge.p2.y * dbu})")
        edge_id += 1

    polygon_id += 1

TOP CELL: test
TOP CELL: cell1
layer index: 0 layer: 2 datatype: 0 name:  info: 2/0
layer index: 1 layer: 1 datatype: 0 name:  info: 1/0
layer index: 2 layer: 3 datatype: 0 name:  info: 3/0
TOP_BBOX: (-7200,-2800;9500,6300)
LAYER_TOP_BBOX: (-6000,500;9500,6300)
box (0,500;1000,1000) r0 *1 0,0
Polygon 0
0.0 0.5
0.0 1.0
1.0 1.0
1.0 0.5
Edge 0:(0.0,0.5)--(0.0,1.0)
Edge 1:(0.0,1.0)--(1.0,1.0)
Edge 2:(1.0,1.0)--(1.0,0.5)
Edge 3:(1.0,0.5)--(0.0,0.5)
box (-6000,1200;9500,1400) r0 *1 0,0
Polygon 1
-6.0 1.2
-6.0 1.4000000000000001
9.5 1.4000000000000001
9.5 1.2
Edge 0:(-6.0,1.2)--(-6.0,1.4000000000000001)
Edge 1:(-6.0,1.4000000000000001)--(9.5,1.4000000000000001)
Edge 2:(9.5,1.4000000000000001)--(9.5,1.2)
Edge 3:(9.5,1.2)--(-6.0,1.2)
